In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import AdaBoostRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import ElasticNet
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
#saved in results\regression-by-source
# eh melhor tirar os ouliers antes de unificar ou depois? tem que ver isso

In [ ]:
# # Here I combine datasets with the same source (e.g., ce-*)
# def combine_datasets(path, substring):
#     files = [f for f in os.listdir(path) if substring in f and f.endswith('.csv')]
#     if not files:
#         print(f"No files containing the substring '{substring}' were found.")
#         return None
#     dataframe_list = []
#     for file in files:
#         file_path = os.path.join(path, file)
#         df = pd.read_csv(file_path)
#         dataframe_list.append(df)
    
#     combined_df = pd.concat(dataframe_list, ignore_index=True)
#     return combined_df

# path = '../../datasets/serie-multivariada'
# dataframes_by_source = {}

# for dirs, root, files in os.walk(path):
#     for file in files:
#         if file.endswith('.csv'):
#             source = '-' + file.split('-')[2] + '-'
#             #print(f"Processing files with source: {source}")
#             df = combine_datasets(path, source)
            
#             if df is not None:
#                 key_name = f"{source.strip('-')}" 
#                 dataframes_by_source[key_name] = df


In [ ]:
# Função para plotar os dados após a remoção de outliers - so pra visualização
def plot_dataframe(dataframe, columns, title="DataFrame After Outlier Removal"):
    plt.figure(figsize=(12, 8))
    for i, column in enumerate(columns):
        if column in dataframe.columns and dataframe[column].dtype in ['int64', 'float64']:
            plt.subplot(len(columns), 1, i + 1)
            sns.histplot(dataframe[column], kde=True, bins=30, color='blue')
            plt.title(f"{title}: {column}", fontsize=12)
            plt.xlabel(column, fontsize=10)
            plt.ylabel("Frequência", fontsize=10)
            plt.grid(True, alpha=0.5)
    plt.tight_layout()
    plt.show()

# Função para combinar datasets
def combine_datasets(path, substring):
    files = [f for f in os.listdir(path) if substring in f and f.endswith('.csv')]
    if not files:
        print(f"No files containing the substring '{substring}' were found.")
        return None
    
    dataframe_list = []
    for file in files:
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        dataframe_list.append(df)
    
    combined_df = pd.concat(dataframe_list, ignore_index=True)
    return combined_df

def remove_outliers(dataframe, columns, n_std=3):
    df_clean = dataframe.copy()
    
    original_size = len(df_clean)
    mask = np.ones(len(df_clean), dtype=bool)
    
    for column in columns:
        if df_clean[column].dtype in ['int64', 'float64']:
            z_scores = np.abs(stats.zscore(df_clean[column], nan_policy='omit'))
            mask = mask & (z_scores < n_std)
    
    df_clean = df_clean[mask]
    
    removed_rows = original_size - len(df_clean)
    removal_percentage = (removed_rows / original_size) * 100
    print(f"Removed {removed_rows} rows ({removal_percentage:.2f}%) as outliers from {dataframe.shape}")
    
    return df_clean

path = '../../datasets/serie-multivariada'
dataframes_by_source = {}

columns_to_check = ['Vazao_bbr', 'Vazao_cubic', 'Atraso(ms)', 'Hop_count']

for dirs, root, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            source = '-' + file.split('-')[2] + '-'
            df = combine_datasets(path, source)
            if df is not None:
                df_clean = remove_outliers(df, columns_to_check)
                key_name = f"{source.strip('-')}"
                dataframes_by_source[key_name] = df_clean
                
                plot_dataframe(df_clean, columns_to_check, title=f"Source: {key_name}")




In [ ]:
def predict_and_evaluate_models(dataframe, target, features, source, rmse_results_dict, prediction_results_folder):
    df_copy = dataframe.copy()
    X = df_copy[features]
    y = df_copy[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
    models = {
        "RandomForestRegressor": RandomForestRegressor(max_depth=10, n_estimators=200, random_state=42),
        "LinearRegression": LinearRegression(),
        "PolynomialRegression": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "AdaBoostRegressor": AdaBoostRegressor(n_estimators=100, random_state=42),
        "XGBRegressor": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "MLPRegressor": MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam', max_iter=1000, random_state=42),
        "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
        "KNeighborsRegressor": KNeighborsRegressor(n_neighbors=3),
        "CatBoostRegressor": CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_state=42, verbose=False),
        "LGBMRegressor": LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=-1, random_state=42),
        "SVR": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    }
    # Store predictions for this source
    predictions = {
        "y_test": y_test.values
    }

    # Initialize results dictionary for this target if it doesn't exist
    if target not in rmse_results_dict:
        rmse_results_dict[target] = []

    rmse_results = {"source": source}  # Initialize with source

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Store prediction values for this model
        predictions[f"y_predict_{model_name}"] = y_pred

        # Calculate RMSE and normalize by the mean of y_test
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        rmse_normalized = rmse / np.mean(y_test)

        # Store the RMSE for each model
        rmse_results[model_name] = rmse_normalized

    # Append RMSE results for the current source and target
    rmse_results_dict[target].append(rmse_results)

    # Save predictions to a CSV
    predictions_df = pd.DataFrame(predictions)
    prediction_file_path = os.path.join(prediction_results_folder, f"{target}_{source}.csv")
    os.makedirs(prediction_results_folder, exist_ok=True)
    predictions_df.to_csv(prediction_file_path, index=False)

    return rmse_results_dict

def save_all_rmse_results(rmse_results_dict, output_folder_base):
    os.makedirs(output_folder_base, exist_ok=True)
    
    # Save results for each target
    for target, results in rmse_results_dict.items():
        # Convert list of dictionaries to DataFrame
        rmse_results_df = pd.DataFrame(results)
        
        # Save RMSE results to CSV
        output_file_rmse = os.path.join(output_folder_base, f"regression_{target}_rmse.csv")
        rmse_results_df.to_csv(output_file_rmse, index=False)

# Example usage
target_columns = ['Vazao_bbr', 'Vazao_cubic']
features = ['Atraso(ms)', 'Hop_count', 'Bottleneck']
folder = '../../results/regression/predictions-regression-bysource'
prediction_results_folder = os.path.join(os.getcwd(), folder)
output_folder_base = os.path.join(os.getcwd(), folder)

# Initialize results dictionary
rmse_results_dict = {}

# Process each target and source
for target in target_columns:
    for source, dataframe in dataframes_by_source.items():
        rmse_results_dict = predict_and_evaluate_models(
            dataframe, 
            target, 
            features, 
            source, 
            rmse_results_dict, 
            prediction_results_folder
        )
save_all_rmse_results(rmse_results_dict, output_folder_base)

In [ ]:
#teste retirando os outliers

def predict_and_evaluate_models(dataframe, target, features, source, rmse_results_dict, prediction_results_folder):
    df_copy = dataframe.copy()
    X = df_copy[features]
    y = df_copy[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

    models = {
        "RandomForestRegressor": RandomForestRegressor(max_depth=10, n_estimators=200, random_state=42),
        "LinearRegression": LinearRegression(),
        "PolynomialRegression": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "AdaBoostRegressor": AdaBoostRegressor(n_estimators=100, random_state=42),
        "XGBRegressor": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "MLPRegressor": MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam', max_iter=1000, random_state=42),
        "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
        "KNeighborsRegressor": KNeighborsRegressor(n_neighbors=3),
        "CatBoostRegressor": CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_state=42, verbose=False),
        "LGBMRegressor": LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=-1, random_state=42),
        "SVR": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    }

    # Store predictions for this source
    predictions = {
        "y_test": y_test.values
    }

    # Initialize results dictionary for this target if it doesn't exist
    if target not in rmse_results_dict:
        rmse_results_dict[target] = []

    rmse_results = {"source": source}  # Initialize with source

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Store prediction values for this model
        predictions[f"y_predict_{model_name}"] = y_pred

        # Calculate RMSE and normalize by the mean of y_test
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        rmse_normalized = rmse / np.mean(y_test)

        # Store the RMSE for each model
        rmse_results[model_name] = rmse_normalized

    # Append RMSE results for the current source and target
    rmse_results_dict[target].append(rmse_results)

    # Save predictions to a CSV
    predictions_df = pd.DataFrame(predictions)
    prediction_file_path = os.path.join(prediction_results_folder, f"{target}_{source}.csv")
    os.makedirs(prediction_results_folder, exist_ok=True)
    predictions_df.to_csv(prediction_file_path, index=False)

    return rmse_results_dict

def save_all_rmse_results(rmse_results_dict, output_folder_base):
    os.makedirs(output_folder_base, exist_ok=True)
    
    # Save results for each target
    for target, results in rmse_results_dict.items():
        # Convert list of dictionaries to DataFrame
        rmse_results_df = pd.DataFrame(results)
        
        # Save RMSE results to CSV
        output_file_rmse = os.path.join(output_folder_base, f"regression_{target}_rmse.csv")
        rmse_results_df.to_csv(output_file_rmse, index=False)


def combine_datasets(path, substring):
    files = [f for f in os.listdir(path) if substring in f and f.endswith('.csv')]
    if not files:
        print(f"No files containing the substring '{substring}' were found.")
        return None
    
    dataframe_list = []
    for file in files:
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        dataframe_list.append(df)
    
    combined_df = pd.concat(dataframe_list, ignore_index=True)
    return combined_df

def remove_outliers(dataframe, columns, n_std=3):
    df_clean = dataframe.copy()
    
    original_size = len(df_clean)
    mask = np.ones(len(df_clean), dtype=bool)
    
    for column in columns:
        if df_clean[column].dtype in ['int64', 'float64']:
            # Calculate z-scores
            z_scores = np.abs(stats.zscore(df_clean[column], nan_policy='omit'))
            # Update mask to exclude outliers
            mask = mask & (z_scores < n_std)
    
    # Apply mask to dataframe
    df_clean = df_clean[mask]
    
    # Calculate and print removal statistics
    removed_rows = original_size - len(df_clean)
    removal_percentage = (removed_rows / original_size) * 100
    print(f"Removed {removed_rows} rows ({removal_percentage:.2f}%) as outliers from {dataframe.shape}")
    
    return df_clean

# Modified main processing code
path = '../../datasets/serie-multivariada'
dataframes_by_source = {}

# Define columns to check for outliers
columns_to_check = ['Vazao_bbr', 'Vazao_cubic', 'Atraso(ms)', 'Hop_count']

for dirs, root, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            source = '-' + file.split('-')[2] + '-'
            df = combine_datasets(path, source)
            if df is not None:
                # Add outlier removal step
                df_clean = remove_outliers(df, columns_to_check)
                key_name = f"{source.strip('-')}"
                dataframes_by_source[key_name] = df_clean

# Rest of your code remains the same...
target_columns = ['Vazao_bbr', 'Vazao_cubic']
features = ['Atraso(ms)', 'Hop_count', 'Bottleneck']
folder = '../../results/regression/predictions-regression-bysource'
prediction_results_folder = os.path.join(os.getcwd(), folder)
output_folder_base = os.path.join(os.getcwd(), folder)

# Initialize results dictionary
rmse_results_dict = {}

# Process each target and source
for target in target_columns:
    for source, dataframe in dataframes_by_source.items():
        rmse_results_dict = predict_and_evaluate_models(
            dataframe,
            target,
            features,
            source,
            rmse_results_dict,
            prediction_results_folder
        )
save_all_rmse_results(rmse_results_dict, output_folder_base)

In [ ]:
# Here is the function that randomly removes values from the target column and imputes the data / without normalizing the data for training

def impute_and_evaluate_models(dataframe, target, features, source, output_csv):
    df_copy = dataframe.copy()

    np.random.seed(42)
    missing_rate = 0.2 
    n_missing = int(len(df_copy) * missing_rate)
    missing_indices = np.random.choice(df_copy.index, n_missing, replace=False)
    df_copy.loc[missing_indices, target] = np.nan

    #normalization_val = df_copy[target].mean()

    df_known = df_copy.dropna(subset=[target])  
    df_missing = df_copy[df_copy[target].isna()]  

    X_known = df_known[features]
    y_known = df_known[target]

    X_missing = df_missing[features]

    models = {
        "RandomForestRegressor": RandomForestRegressor(max_depth=10, n_estimators=200, random_state=42),
        "LinearRegression": LinearRegression(),
        "PolynomialRegression": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "AdaBoostRegressor": AdaBoostRegressor(n_estimators=100, random_state=42),
        "XGBRegressor": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "MLPRegressor": MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam', max_iter=1000, random_state=42),
        "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
        "KNeighborsRegressor": KNeighborsRegressor(n_neighbors=5),
        "CatBoostRegressor": CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_state=42, verbose=False),
        "LGBMRegressor": LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=-1, random_state=42),
        "SVR": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    }

    results = {"source": [source]}
    for model_name, model in models.items():
        model.fit(X_known, y_known)
        imputed = model.predict(X_missing)
        df_copy.loc[df_missing.index, target] = imputed
        original_values = dataframe.loc[missing_indices, target]  # Original values
        normalization_val = original_values.mean()
        imputed_values = df_copy.loc[missing_indices, target]  # Imputed values

        mse = mean_squared_error(original_values, imputed_values)
        rmse = np.sqrt(mse) / normalization_val
        results[model_name] = [rmse]

    df_results = pd.DataFrame(results)

    if os.path.exists(output_csv):
        existing_df = pd.read_csv(output_csv)
        df_results = pd.concat([existing_df, df_results], ignore_index=True)
    df_results.to_csv(output_csv, index=False)

    return df_results

target_columns = ['Vazao_bbr', 'Vazao_cubic']
features = ['Atraso(ms)', 'Hop_count', 'Bottleneck']  # 'Timestamp_cubic'

for target in target_columns:
    for key, value in dataframes_by_source.items():
        archive = "errorFinding-imputation-" + target + ".csv"
        source = key
        dataframe = dataframes_by_source[key]
        results = impute_and_evaluate_models(dataframe, target, features, source, archive)


In [ ]:
#prevendo os valores do datasets separadamente 
def combine_datasets(path, substring):
    files = [f for f in os.listdir(path) if substring in f and f.endswith('.csv')]
    if not files:
        print(f"No files containing the substring '{substring}' were found.")
        return None
    
    dataframe_list = []
    for file in files:
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        dataframe_list.append(df)
    
    combined_df = pd.concat(dataframe_list, ignore_index=True)
    return combined_df

def save_all_rmse_results(rmse_results_dict, output_folder_base):
    os.makedirs(output_folder_base, exist_ok=True)
    
    # Save results for each target
    for target, results in rmse_results_dict.items():
        # Convert list of dictionaries to DataFrame
        rmse_results_df = pd.DataFrame(results)
        
        # Save RMSE results to CSV
        output_file_rmse = os.path.join(output_folder_base, f"regression_{target}_rmse.csv")
        rmse_results_df.to_csv(output_file_rmse, index=False)

def remove_outliers(dataframe, columns, n_std=3):
    df_clean = dataframe.copy()
    
    original_size = len(df_clean)
    mask = np.ones(len(df_clean), dtype=bool)
    
    for column in columns:
        if df_clean[column].dtype in ['int64', 'float64']:
            # Calculate z-scores
            z_scores = np.abs(stats.zscore(df_clean[column], nan_policy='omit'))
            # Update mask to exclude outliers
            mask = mask & (z_scores < n_std)
    
    # Apply mask to dataframe
    df_clean = df_clean[mask]
    
    # Calculate and print removal statistics
    removed_rows = original_size - len(df_clean)
    removal_percentage = (removed_rows / original_size) * 100
    print(f"Removed {removed_rows} rows ({removal_percentage:.2f}%) as outliers from {dataframe.shape}")
    
    return df_clean

def predict_and_evaluate_models_with_related(dataframe, target, features, source, rmse_results_dict, 
                                             prediction_results_folder, related_datasets_folder):
    df_copy = dataframe.copy()
    X = df_copy[features]
    y = df_copy[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

    models = {
        "RandomForestRegressor": RandomForestRegressor(max_depth=10, n_estimators=200, random_state=42),
        "LinearRegression": LinearRegression(),
        "PolynomialRegression": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "AdaBoostRegressor": AdaBoostRegressor(n_estimators=100, random_state=42),
        "XGBRegressor": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "MLPRegressor": MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam', max_iter=1000, random_state=42),
        "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
        "KNeighborsRegressor": KNeighborsRegressor(n_neighbors=3),
        "CatBoostRegressor": CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_state=42, verbose=False),
        "LGBMRegressor": LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=-1, random_state=42),
        "SVR": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    }

    # Store predictions for this source
    predictions = {
        "y_test": y_test.values
    }

    if target not in rmse_results_dict:
        rmse_results_dict[target] = []

    rmse_results = {"source": source}  # Initialize with source

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Store prediction values for this model
        predictions[f"y_predict_{model_name}"] = y_pred

        # Calculate RMSE and normalize by the mean of y_test
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        rmse_normalized = rmse / np.mean(y_test)

        rmse_results[model_name] = rmse_normalized

        # Process related datasets for additional predictions
        related_files = [f for f in os.listdir(related_datasets_folder) if source in f and f.endswith('.csv')]

        for related_file in related_files:
            related_file_path = os.path.join(related_datasets_folder, related_file)
            related_df = pd.read_csv(related_file_path)

            # Ensure related_df has the required features
            if all(feature in related_df.columns for feature in features):
                X_related = related_df[features]
                y_related_actual = related_df[target]

                # Predict on related dataset
                y_related_pred = model.predict(X_related)

                # Calculate RMSE and normalize
                rmse_related = np.sqrt(mean_squared_error(y_related_actual, y_related_pred))
                rmse_related_normalized = rmse_related / np.mean(y_related_actual)

                rmse_results[f"{model_name}_related_{related_file}"] = rmse_related_normalized

                # Save related dataset predictions
                related_predictions_df = pd.DataFrame({
                    "y_actual": y_related_actual,
                    f"y_predict_{model_name}": y_related_pred
                })
                related_output_path = os.path.join(prediction_results_folder, f"{target}_{source}_related_{related_file}.csv")
                os.makedirs(prediction_results_folder, exist_ok=True)
                related_predictions_df.to_csv(related_output_path, index=False)

    # Append RMSE results for the current source and target
    rmse_results_dict[target].append(rmse_results)

    # Save predictions to a CSV
    predictions_df = pd.DataFrame(predictions)
    prediction_file_path = os.path.join(prediction_results_folder, f"{target}_{source}.csv")
    os.makedirs(prediction_results_folder, exist_ok=True)
    predictions_df.to_csv(prediction_file_path, index=False)

    return rmse_results_dict

# Main processing code
path = '../../datasets/serie-multivariada'
related_datasets_folder = '../../datasets/serie-multivariada'
dataframes_by_source = {}
columns_to_check = ['Vazao_bbr', 'Vazao_cubic', 'Atraso(ms)', 'Hop_count']

for dirs, root, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            source = '-' + file.split('-')[2] + '-'
            df = combine_datasets(path, source)
            if df is not None:
                df_clean = remove_outliers(df, columns_to_check)
                key_name = f"{source.strip('-')}"
                dataframes_by_source[key_name] = df_clean

target_columns = ['Vazao_bbr', 'Vazao_cubic']
features = ['Atraso(ms)', 'Hop_count', 'Bottleneck']
folder = '../../results/regression/Teste-predictions-related'
prediction_results_folder = os.path.join(os.getcwd(), folder)
output_folder_base = os.path.join(os.getcwd(), folder)

# Initialize results dictionary
rmse_results_dict = {}

for target in target_columns:
    for source, dataframe in dataframes_by_source.items():
        rmse_results_dict = predict_and_evaluate_models_with_related(
            dataframe,
            target,
            features,
            source,
            rmse_results_dict,
            prediction_results_folder,
            related_datasets_folder
        )

save_all_rmse_results(rmse_results_dict, output_folder_base)

TESTE - EM CONSTRUÇÃO 


In [ ]:
# retirando 20% dos dados do meu dataset original (antes de combina-los) para posterior analise do rmse 
#tstando
import os
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.neural_network import MLPRegressor

def split_and_save_original_data(path, substring):
    files = [f for f in os.listdir(path) if f"-{substring}-" in f and f.endswith('.csv')]
    if not files:
        print(f"No files containing the substring '-{substring}-' were found.")
        return None, {}
    
    train_dataframes = []
    test_data_dict = {}
    
    for file in files:
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        
        # Split each individual dataset
        train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
        
        # Store test data with file name as key
        test_data_dict[file] = test_df
        train_dataframes.append(train_df)
    
    # Combine only training portions
    combined_train = pd.concat(train_dataframes, ignore_index=True)
    return combined_train, test_data_dict

def save_all_rmse_results(rmse_results_dict, output_folder_base):
    os.makedirs(output_folder_base, exist_ok=True)
    for target, results in rmse_results_dict.items():
        rmse_results_df = pd.DataFrame(results)
        output_file_rmse = os.path.join(output_folder_base, f"regression_{target}_rmse.csv")
        rmse_results_df.to_csv(output_file_rmse, index=False)

def remove_outliers(dataframe, columns, n_std=3):
    df_clean = dataframe.copy()
    original_size = len(df_clean)
    mask = np.ones(len(df_clean), dtype=bool)
    for column in columns:
        if df_clean[column].dtype in ['int64', 'float64']:
            z_scores = np.abs(stats.zscore(df_clean[column], nan_policy='omit'))
            mask = mask & (z_scores < n_std)
    df_clean = df_clean[mask]
    removed_rows = original_size - len(df_clean)
    removal_percentage = (removed_rows / original_size) * 100
    print(f"Removed {removed_rows} rows ({removal_percentage:.2f}%) as outliers from {dataframe.shape}")
    return df_clean

################################################################

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

def predict_and_evaluate_models_with_related(train_dataframe, test_data_dict, target, features, source, 
                                           rmse_results_dict, prediction_results_folder):
    # Prepare training data
    X_train = train_dataframe[features]
    y_train = train_dataframe[target]

    # Embaralha os dados de treinamento
    X_train, _, y_train, _ = train_test_split(X_train, y_train, test_size=0, shuffle=True, random_state=42)

    models = {
        "RandomForestRegressor": RandomForestRegressor(max_depth=10, n_estimators=200, random_state=42),
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "XGBRegressor": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        "CatBoostRegressor": CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_state=42, verbose=False),
        "LGBMRegressor": LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=-1, random_state=42)
    }

    if target not in rmse_results_dict:
        rmse_results_dict[target] = []

    rmse_results = {"source": source}
    
    # Process each original test file
    for file_name, test_df in test_data_dict.items():
        X_test = test_df[features]
        y_test = test_df[target]
        
        # Initialize predictions DataFrame with actual values
        predictions_df = pd.DataFrame({"y_actual": y_test.values})
        
        # Train and predict with each model
        for model_name, model in models.items():
            print(f"Training {model_name} for {file_name}")
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            
            # Add predictions to DataFrame
            predictions_df[f"y_predict_{model_name}"] = y_pred
            
            # Calculate and store RMSE
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            rmse_normalized = rmse / np.mean(y_test)
            rmse_results[f"{model_name}_{file_name}"] = rmse_normalized
        
        # Save predictions for this test file
        prediction_file_path = os.path.join(prediction_results_folder, f"{target}_{source}_{file_name}_predictions.csv")
        os.makedirs(prediction_results_folder, exist_ok=True)
        predictions_df.to_csv(prediction_file_path, index=False)

    rmse_results_dict[target].append(rmse_results)
    return rmse_results_dict

# def predict_and_evaluate_models_with_related(train_dataframe, test_data_dict, target, features, source, 
#                                            rmse_results_dict, prediction_results_folder):
#     # Prepare training data
#     X_train = train_dataframe[features]
#     y_train = train_dataframe[target]

#     models = {
#         "RandomForestRegressor": RandomForestRegressor(max_depth=10, n_estimators=200, random_state=42),
#         "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
#         "XGBRegressor": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
#         "CatBoostRegressor": CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_state=42, verbose=False),
#         "LGBMRegressor": LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=-1, random_state=42)
#     }

#     if target not in rmse_results_dict:
#         rmse_results_dict[target] = []

#     rmse_results = {"source": source}
    
#     # Process each original test file
#     for file_name, test_df in test_data_dict.items():
#         X_test = test_df[features]
#         y_test = test_df[target]
        
#         # Initialize predictions DataFrame with actual values
#         predictions_df = pd.DataFrame({"y_actual": y_test.values})
        
#         # Train and predict with each model
#         for model_name, model in models.items():
#             print(f"Training {model_name} for {file_name}")
#             model.fit(X_train, y_train)
#             y_pred = model.predict(X_test)
            
#             # Add predictions to DataFrame
#             predictions_df[f"y_predict_{model_name}"] = y_pred
            
#             # Calculate and store RMSE
#             rmse = np.sqrt(mean_squared_error(y_test, y_pred))
#             rmse_normalized = rmse / np.mean(y_test)
#             rmse_results[f"{model_name}_{file_name}"] = rmse_normalized
        
#         # Save predictions for this test file
#         prediction_file_path = os.path.join(prediction_results_folder, f"{target}_{source}_{file_name}_predictions.csv")
#         os.makedirs(prediction_results_folder, exist_ok=True)
#         predictions_df.to_csv(prediction_file_path, index=False)

#     rmse_results_dict[target].append(rmse_results)
#     return rmse_results_dict

# Código principal
path = '../../datasets/serie-multivariada'
dataframes_by_source = {}
test_data_by_source = {}
columns_to_check = ['Vazao_bbr', 'Vazao_cubic', 'Atraso(ms)', 'Hop_count']

# Primeiro, separar e salvar os dados de teste
for dirs, root, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            source = file.split('-')[2]
            train_df, test_dict = split_and_save_original_data(path, source)
            
            if train_df is not None:
                train_df_clean = remove_outliers(train_df, columns_to_check)
                dataframes_by_source[source] = train_df_clean
                test_data_by_source[source] = test_dict

target_columns = ['Vazao_bbr']
features = ['Atraso(ms)', 'Hop_count', 'Bottleneck']
folder = '../../results/regression/teste-predictions-related'
prediction_results_folder = os.path.join(os.getcwd(), folder)
output_folder_base = os.path.join(os.getcwd(), folder)

rmse_results_dict = {}

for target in target_columns:
    for source, train_df in dataframes_by_source.items():
        rmse_results_dict = predict_and_evaluate_models_with_related(
            train_df,
            test_data_by_source[source],
            target,
            features,
            source,
            rmse_results_dict,
            prediction_results_folder
        )

save_all_rmse_results(rmse_results_dict, output_folder_base)